```
<04_more_apples_yolov8.ipynb>

제미나이 의존도: 30-40%

3600여개의 AI Hub 사과 데이터를 사용했다. 저번과는 다르게 train / val / test도 분리했다.
처음에 train_test_split를 쓰려고 했으나, 잘 안 될거 같아서 random.shuffle() 함수를 사용했다.
8:1:1로 나눴다.
결과를 보면 알겠지만.. 214m 34s. 약 3시간 30분이나 걸렸다.
객체탐지는 에폭이 50이나 100으로 시작하는게 국룰이라길래 에폭을 50으로 설정했다.
역시 오래 걸린만큼 테스트 이미지를 넣어보니 conf는 1.0이나 0.99가 나왔다.

한가지 아쉬운 점은, 학습에 사용된 사과는 픽셀 안에 사과만 덩그러니 있다는 것이다.
그래서 사과가 여러 개 있는 사진을 올려도 가장 큰 네모박스 하나밖에 못 치는 상황이다.
내가 원한 건 여러 개의 네모박스인데 말이다.

그래서.. 사과가 여러 개 있는 이미지를 구해서 작업해보고싶다는 생각이 들었다.
```

<목표> 이미지 약 3600장을 train / val / test로 분리해서 YOLOv8 모델에 학습시키기
1. 이미지 경로 리스트를 담고 있는 객체를 생성한다.
2. 

In [36]:
import os
import shutil
from pathlib import Path
# from sklearn.model_selection import train_test_split
import random
import json
import time

In [27]:
# .ipynb_checkpoints 제거 함수
def cleaner(dataset_root):
    print("데이터셋 내부의 .ipynb_checkpoints 청소 시작.")
    for root, dirs, files in os.walk(dataset_root):
        if ".ipynb_checkpoints" in dirs:
            target_dir = os.path.join(root, ".ipynb_checkpoints")
            shutil.rmtree(target_dir)
            print(f"청소 완료: {target_dir}")

In [45]:
def copy_image_and_convert_json_to_txt(
    target_images, image_path, json_path, dest_image_path, dest_label_path
):
    start_time = time.time()
    copied_count = 0
    
    for image in target_images:
        if image.endswith(('jpg', 'jpeg', 'png')):
            # print(os.path.join(image_path, image)) # images/apple_from_aihub/../apple001.png
            image_full_path = os.path.join(image_path, image)
            is_image_exists = os.path.isfile(image_full_path)
            
            pure_img_name = Path(image).stem
            # print(pure_img_name) # apple001
            json_file_name = f"{pure_img_name}.json" # apple001.json
            json_full_path = os.path.join(json_path, json_file_name)
            is_json_exists = os.path.isfile(json_full_path)
    
            if is_image_exists and is_json_exists:
                # 이미지 복사
                shutil.copy(image_full_path, dest_image_path)
                # json을 txt로 변환
                with open(json_full_path, encoding="utf-8") as f:
                    data = json.load(f)
    
                    img_height = data["img_height"]
                    img_width = data["img_width"]
                    bndbox = data["bndbox"]
                    xmin = bndbox["xmin"]
                    ymin = bndbox["ymin"]
                    xmax = bndbox["xmax"]
                    ymax = bndbox["ymax"]
    
                    # print(xmin, xmax)
                    px_cx = (xmin + xmax) / 2
                    px_cy = (ymin + ymax) / 2
                    px_w = (xmax - xmin)
                    px_h = (ymax - ymin)
    
                    yolo_cx = px_cx / img_width
                    yolo_cy = px_cy / img_height
                    yolo_w = px_w / img_width
                    yolo_h = px_h / img_height
    
                    # print(f"[0 {yolo_cx} {yolo_cy} {yolo_w} {yolo_h}]")
                    save_path = os.path.join(dest_label_path, f"{pure_img_name}.txt")
                    with open(save_path, "w", encoding="utf-8") as f:
                        f.write(f"0 {yolo_cx} {yolo_cy} {yolo_w} {yolo_h}")

                copied_count += 1
                        
    end_time = time.time()
    elapsed_time = end_time - start_time
    mins = int(elapsed_time // 60)
    secs = int(elapsed_time % 60)
    
    print(f"대상 이미지 경로: {dest_image_path}, 복사된 이미지 수: {copied_count}, 소요시간: {mins}m:{secs}s")

In [15]:
# 디렉토리 생성
common_dest_path = "images/more_apples"

dest_train_image_path = os.path.join(common_dest_path, "train/images")
dest_train_label_path = os.path.join(common_dest_path, "train/labels")
dest_val_image_path = os.path.join(common_dest_path, "val/images")
dest_val_label_path = os.path.join(common_dest_path, "val/labels")
dest_test_image_path = os.path.join(common_dest_path, "test/images")
dest_test_label_path = os.path.join(common_dest_path, "test/labels")

dir_paths = []
dir_paths.append(dest_train_image_path)
dir_paths.append(dest_train_label_path)
dir_paths.append(dest_val_image_path)
dir_paths.append(dest_val_label_path)
dir_paths.append(dest_test_image_path)
dir_paths.append(dest_test_label_path)

for dir_path in dir_paths:
    os.makedirs(dir_path, exist_ok=True)

In [29]:
# train / val / test 분할
common_path = "images/apple_from_aihub/01.데이터/1.Training"

image_path = os.path.join(common_path, "원천데이터_230921_add/Apple_fuji_L")
json_path = os.path.join(common_path, "라벨링데이터_230921_add/Apple_fuji_L")

images = os.listdir(image_path)
# print(len(images)) # 3649
random.seed(42)
random.shuffle(images)
# print(images[:5])

split_idx = int(len(images) * 0.8)
train_images = images[:split_idx]
val_test_images = images[split_idx:]

split_idx_for_val_test = int(len(val_test_images) * 0.5)
val_images = val_test_images[:split_idx_for_val_test]
test_images = val_test_images[split_idx_for_val_test:]

# print(len(images), split_idx)
print(f"{len(train_images)}, {len(val_images)}, {len(test_images)}")

2919, 365, 365


In [46]:
# train / val / test 이미지 복사 및 txt 생성
copy_image_and_convert_json_to_txt(
    train_images, image_path, json_path, dest_train_image_path, dest_train_label_path
)
copy_image_and_convert_json_to_txt(
    val_images, image_path, json_path, dest_val_image_path, dest_val_label_path
)
copy_image_and_convert_json_to_txt(
    test_images, image_path, json_path, dest_test_image_path, dest_test_label_path
)


대상 이미지 경로: images/more_apples/train/images, 복사된 이미지 수: 2919, 소요시간: 0m:6s
대상 이미지 경로: images/more_apples/val/images, 복사된 이미지 수: 365, 소요시간: 0m:0s
대상 이미지 경로: images/more_apples/test/images, 복사된 이미지 수: 365, 소요시간: 0m:0s


In [31]:
# data.yaml 파일 생성
with open(os.path.join(common_dest_path, "data.yaml"), "w", encoding="utf-8") as f:
    f.write("train: ../train/images\n")
    f.write("val: ../val/images\n")
    f.write("\n")
    f.write("nc: 1\n")
    f.write("names: ['apple']")

In [47]:
cleaner(common_dest_path)

데이터셋 내부의 .ipynb_checkpoints 청소 시작.
청소 완료: images/more_apples/.ipynb_checkpoints
청소 완료: images/more_apples/test/images/.ipynb_checkpoints
청소 완료: images/more_apples/test/labels/.ipynb_checkpoints
청소 완료: images/more_apples/train/images/.ipynb_checkpoints
청소 완료: images/more_apples/train/labels/.ipynb_checkpoints
청소 완료: images/more_apples/val/images/.ipynb_checkpoints
청소 완료: images/more_apples/val/labels/.ipynb_checkpoints


In [51]:
from ultralytics import YOLO

current_dir = os.getcwd()

model = YOLO("yolov8n.pt")

start_time = time.time()

results = model.train(
    data=os.path.join(common_dest_path, "data.yaml"), 
    epochs=50, 
    imgsz=640, 
    device="mps", 

    batch=32, # 한 번에 32장씩 묶어서 GPU에 던진다. (기본보다 훨씬 빨라진다.)
    workers=4, # CPU가 데이터를 미리 읽어오는 일꾼의 수. (맥os 칩 성능 활용)

    project=os.path.join(current_dir, "runs"), 
    name="more_apples"
)

end_time = time.time()

elapsed_time = end_time - start_time
mins = int(elapsed_time // 60)
secs = int(elapsed_time % 60)
print(f"{mins}m {secs}s")

New https://pypi.org/project/ultralytics/8.4.89 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.87 🚀 Python-3.11.15 torch-2.12.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=images/more_apples/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=more_apples-3, nb

In [53]:
model = YOLO("runs/more_apples/weights/best.pt")

test_img_path = "images/more_apples/test/images/apple_fuji_L_1-105.png"

result = model.predict(
    source=test_img_path, 
    save=True, 
    project=os.path.join(current_dir, "runs"), 
    name="test_prediction"
)


image 1/1 /Users/jeongjaehun/Github/03_object_detection/images/more_apples/test/images/apple_fuji_L_1-105.png: 640x640 1 apple, 50.9ms
Speed: 2.6ms preprocess, 50.9ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /Users/jeongjaehun/Github/03_object_detection/runs/test_prediction
